# **05 - Build `venues` table (SeatGeek + Census) for Gametime shows**
***

**What:** Adds information about each *venue* (country and state, capacity, time zone, county income) so the model can use it to predict ticket prices.

**Why:** Gametime tells us the venue's name and city, but nothing about the venue itself. Price depends a lot on venue size: the same artist costs far more in an arena than in a club. Gametime also has no state or country, so this notebook is how we tell US shows from non-US ones.

**What it does:**
1. Lists the venues in `data/shows.csv` (about 3,000), with how many shows each has.
2. Matches each venue to **SeatGeek** by name and city, which gives country, state, latitude and longitude, time zone, capacity and a venue popularity score.
3. Uses the coordinates to look up the county on the **Census** geocoder, then adds county median income and population (optional).
4. Fills in capacity gaps: SeatGeek only has capacity for a small share of venues, so the busiest venues missing a capacity go into a short hand-fill file, and a proxy (typical maximum number of listings per show) is saved for everyone.
5. Saves one row per venue to `data/venues.csv`.

**Where it fits:** it runs after `03_build_shows.ipynb` (to-do step 3, with step 4 for capacity). The model table (step 7) joins `venues` to `shows` on `venue_key`, and uses `is_us` to keep only US shows.

---

**Input:** `data/shows.csv` (from `03_build_shows.ipynb`)
**Output:** `data/venues.csv`, one row per `venue_key`, and `data/manual/venue_capacity_manual.csv` (a list of busy venues to fill in by hand)

**Before running**, `.env` needs `SG_CLIENT_ID=...`. Census needs no key for this many calls (`CENSUS_API_KEY` is optional).

**Reuses old lookups.** Answers are saved in `data/gametime/interim/`, so stopping and re-running only fetches what is missing. (The old Ticketmaster venue lookups are keyed by Ticketmaster IDs, so they cannot be reused here.)

**Runtime:** about 20-40 minutes for ~3,000 venues (SeatGeek is the slow part; Census adds about 10 minutes). Set `LIMIT = 20` in the next cell to try it on a few venues first.

## 1. Setup

In [1]:
import json
import math
import os
import re
import time
import unicodedata
from difflib import SequenceMatcher
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv(".env", override=True)
SG_ID = os.getenv("SG_CLIENT_ID")
CENSUS_KEY = os.getenv("CENSUS_API_KEY")       # optional
assert SG_ID, "Add SG_CLIENT_ID to .env"

LIMIT = None           # e.g. 20 to test on the busiest few venues first
USE_CENSUS = True      # county income / population (US venues only)
ACS_YEAR = 2023        # ACS 5-year release; try 2024 if it's available
MANUAL_TOP_N = 150     # busiest venues with no capacity to hand-fill

SHOWS_PATH = Path("data/shows.csv")
OUT_PATH = Path("data/venues.csv")
INTERIM = Path("data/gametime/interim")
MANUAL = Path("data/manual")
for p in (INTERIM, MANUAL):
    p.mkdir(parents=True, exist_ok=True)
SG_CACHE = INTERIM / "seatgeek_venues.jsonl"
GEO_CACHE = INTERIM / "census_geocoder.jsonl"
SG = "https://api.seatgeek.com/2"
print("SeatGeek ID found:", bool(SG_ID))

SeatGeek ID found: True


## 2. One row per venue from the shows table
`venue_key` (venue name + city) is the join key. `median_max_listings` is the typical maximum number of listings a show at this venue had: a rough stand-in for venue size when we have no capacity.

In [2]:
assert SHOWS_PATH.exists(), "Run 03_build_shows.ipynb first (it writes data/shows.csv)"
shows = pd.read_csv(SHOWS_PATH, low_memory=False)

def most_common(s):
    return s.mode().iat[0] if not s.mode().empty else s.iat[0]

venues = (shows.groupby("venue_key")
               .agg(venue=("venue", most_common), city=("city", most_common),
                    n_shows=("event_id", "size"),
                    median_price_min=("price_min", "median"),
                    median_max_listings=("max_listings", "median"))
               .reset_index()
               .sort_values("n_shows", ascending=False))
if LIMIT:
    venues = venues.head(LIMIT)
print(f"{len(venues):,} venues; top 150 hold {venues['n_shows'].head(150).sum() / venues['n_shows'].sum():.0%} of shows")
venues.head(10)

2,722 venues; top 150 hold 25% of shows


,venue_key,venue,city,n_shows,median_price_min,median_max_listings
1493,masquerade|atlanta,Masquerade,Atlanta,115,44.0,1.0
2015,red rocks amphitheatre|morrison,Red Rocks Amphitheatre,Morrison,100,97.5,27.5
1180,house of blues|houston,House of Blues,Houston,93,55.0,1.0
2655,westgate resort and casino|las vegas,Westgate Resort & Casino,Las Vegas,89,91.0,25.0
290,blue note|new york,Blue Note,New York,89,110.0,1.0
2709,yoshi s oakland|oakland,Yoshi's Oakland,Oakland,82,123.0,1.0
271,birdland|new york,Birdland,New York,81,188.0,1.0
1028,grand ole opry house|nashville,Grand Ole Opry House,Nashville,80,81.0,77.0
1185,house of blues|san diego,House of Blues,San Diego,78,42.0,1.0
22,9 30 club|washington,9:30 Club,Washington,77,60.0,1.0


## 3. Helpers

In [3]:
def load_cache(path, key):
    if not path.exists():
        return {}
    with path.open(encoding="utf-8") as f:
        return {r[key]: r for r in (json.loads(l) for l in f if l.strip())}


def append_cache(path, row):
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(row) + "\n")


def get_json(url, params, pause=0.3, tries=4):
    for attempt in range(tries):
        try:
            r = requests.get(url, params=params, timeout=30)
        except requests.RequestException:
            time.sleep(2 * (attempt + 1))
            continue
        time.sleep(pause)
        if r.status_code in (429, 500, 502, 503):
            time.sleep(3 * (attempt + 1))
            continue
        if r.status_code in (401, 403):
            raise RuntimeError(f"Access refused ({r.status_code}): check the key. {r.text[:150]}")
        try:
            return r.json()
        except ValueError:
            return None
    return None


STOP = r"\b(the|presented by .*|at .*|music hall|theatre|theater|club|venue|live|ballroom|lounge|bar|tavern)\b"

def norm_venue(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    s = re.sub(r"[^a-z0-9 ]+", " ", s.replace("&", " and "))
    core = re.sub(STOP, " ", s)
    return re.sub(r"\s+", " ", core if core.strip() else s).strip()


def name_sim(a, b):
    return SequenceMatcher(None, norm_venue(a), norm_venue(b)).ratio()


def norm_city(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    s = re.sub(r"[^a-z0-9 ]+", " ", s)
    s = re.sub(r"\bst\b", "saint", s)
    s = re.sub(r"\bmt\b", "mount", s)
    s = re.sub(r"\bft\b", "fort", s)
    return re.sub(r"\s+", " ", s).strip()

## 4. Match each venue to SeatGeek

Gametime gives only a name and a city, so for each venue we search SeatGeek by name **within that city**. If that finds nothing we search by name alone. A candidate is accepted when the names are close (similarity 0.8+) **and the city matches**. The city check matters for chains: "House of Blues" is a perfect name match in a dozen cities.

SeatGeek returns the country and state, which is how we flag non-US venues.

In [4]:
def best_match(v, cands):
    best = None
    for c in cands:
        s = name_sim(v.venue, c.get("name", ""))
        if s >= 0.8 and norm_city(c.get("city", "")) == norm_city(v.city) and (best is None or s > best["sim"]):
            best = {"sim": s, "venue": c}
    return best


def match_venue(v):
    for with_city in (True, False):
        params = {"client_id": SG_ID, "q": v.venue, "per_page": 10}
        if with_city:
            params["city"] = v.city
        m = best_match(v, (get_json(f"{SG}/venues", params) or {}).get("venues", []))
        if m:
            return m
    return None


cache = load_cache(SG_CACHE, "venue_key")
todo = venues[~venues["venue_key"].isin(cache)]
print(f"SeatGeek venues: {len(cache)} cached, {len(todo)} to match (~{len(todo) * 0.8 / 60:.0f} min)")
t0 = time.time()
for i, v in enumerate(todo.itertuples(), 1):
    m = match_venue(v)
    c = m["venue"] if m else {}
    loc = c.get("location") or {}
    append_cache(SG_CACHE, {
        "venue_key": v.venue_key, "matched": bool(m),
        "sg_venue_id": c.get("id"), "sg_name": c.get("name"), "sg_city": c.get("city"),
        "sg_state": c.get("state"), "sg_country": c.get("country"),
        "lat": loc.get("lat"), "lon": loc.get("lon"), "timezone": c.get("timezone"),
        "sg_capacity": c.get("capacity") or None,          # SeatGeek uses 0 for unknown
        "sg_venue_score": c.get("score"), "sg_upcoming_events": c.get("num_upcoming_events"),
        "match_sim": m["sim"] if m else None})
    if i % 50 == 0:
        print(f"  {i}/{len(todo)} | {(time.time() - t0) / 60:.1f} min")

sgv = pd.DataFrame(load_cache(SG_CACHE, "venue_key").values())
sgv = sgv[sgv["venue_key"].isin(venues["venue_key"])]
w = venues.merge(sgv, on="venue_key")
print(f"Matched: {sgv['matched'].mean():.0%} of venues, {w.loc[w['matched'], 'n_shows'].sum() / w['n_shows'].sum():.0%} of shows")
print(f"With capacity: {sgv['sg_capacity'].notna().mean():.0%} of venues")
print(sgv["sg_country"].value_counts(dropna=False).head(8).to_string())

SeatGeek venues: 2722 cached, 0 to match (~0 min)
Matched: 78% of venues, 75% of shows
With capacity: 17% of venues
sg_country
US           1943
NaN           591
Canada        131
UK             27
Germany         9
                3
Australia       3
Ireland         2


In [5]:
# Spot-check the weakest accepted matches: Gametime name vs SeatGeek name
venues.merge(sgv, on="venue_key")[lambda d: d["matched"]] \
      .sort_values("match_sim")[["venue", "city", "sg_name", "sg_city", "match_sim", "sg_capacity"]].head(15)

,venue,city,sg_name,sg_city,match_sim,sg_capacity
2686,Mo's Place,Katy,Mo's Place - Katy,Katy,0.8,NaN
2678,Warner Theatre,Torrington,Warner Theatre - CT,Torrington,0.8,NaN
2464,Castle Knoll Farms,Paoli,Castle Knoll,Paoli,0.8,NaN
2446,CEFCU Center Stage at The Landing,Peoria,Center Stage At The Landing,Peoria,0.8,NaN
1733,Adler Theatre,Davenport,Alder Theatre,Davenport,0.8,NaN
1841,Long Beach Terrace Theater,Long Beach,Long Beach Center Theater,Long Beach,0.8,NaN
1975,The Pavilion at Ole Miss,Oxford,SJB Pavilion at Ole Miss,Oxford,0.8,9500.0
66,The Anthem,Washington,The Anthem - DC,Washington,0.8,NaN
1724,Hiliard Center,Corpus Christi,Hilliard Center Arena,Corpus Christi,0.8,10000.0
1757,Funny Bone Liberty,Liberty Township,Funny Bone - Liberty Township,Liberty Township,0.8,NaN


## 5. Census: county income and population (optional)
The Census geocoder turns each US venue's latitude and longitude (from SeatGeek) into a county code; one ACS call then gives every county's median household income and population.

In [6]:
if USE_CENSUS:
    us = sgv[sgv["matched"] & (sgv["sg_country"] == "US") & sgv["lat"].notna()]
    geo = load_cache(GEO_CACHE, "venue_key")
    todo = us[~us["venue_key"].isin(geo)]
    print(f"Geocoder: {len(geo)} cached, {len(todo)} to fetch")
    for i, v in enumerate(todo.itertuples(), 1):
        js = get_json("https://geocoding.geo.census.gov/geocoder/geographies/coordinates",
                      {"x": v.lon, "y": v.lat, "benchmark": "Public_AR_Current",
                       "vintage": "Current_Current", "layers": "Counties", "format": "json"}, pause=0.2)
        counties = (((js or {}).get("result") or {}).get("geographies") or {}).get("Counties") or []
        append_cache(GEO_CACHE, {"venue_key": v.venue_key, "county_fips": counties[0]["GEOID"] if counties else None})
        if i % 100 == 0:
            print(f"  {i}/{len(todo)}")
    geo = pd.DataFrame(load_cache(GEO_CACHE, "venue_key").values(), columns=["venue_key", "county_fips"])

    params = {"get": "NAME,B19013_001E,B01003_001E", "for": "county:*"}
    if CENSUS_KEY:
        params["key"] = CENSUS_KEY
    acs_rows = get_json(f"https://api.census.gov/data/{ACS_YEAR}/acs/acs5", params)
    acs = pd.DataFrame(acs_rows[1:], columns=acs_rows[0])
    acs["county_fips"] = acs["state"] + acs["county"]
    acs = acs.rename(columns={"NAME": "county_name", "B19013_001E": "median_income", "B01003_001E": "county_population"})
    acs[["median_income", "county_population"]] = acs[["median_income", "county_population"]].apply(pd.to_numeric, errors="coerce")
    acs.loc[acs["median_income"] < 0, "median_income"] = np.nan     # Census uses negative codes for missing
    census = geo.merge(acs[["county_fips", "county_name", "median_income", "county_population"]],
                       on="county_fips", how="left")
    census = census[census["venue_key"].isin(venues["venue_key"])]
    print(f"Counties found: {census['county_fips'].notna().mean():.0%} | with income: {census['median_income'].notna().mean():.0%}")
else:
    census = pd.DataFrame(columns=["venue_key", "county_fips", "county_name", "median_income", "county_population"])

Geocoder: 1943 cached, 0 to fetch
Counties found: 100% | with income: 100%


## 6. Venue capacity: hand-fill the busiest venues
SeatGeek has a capacity for only a small share of venues, so we fill the gap two ways:
- **Hand-fill:** the first full run writes `data/manual/venue_capacity_manual.csv` listing the busiest venues with no capacity (the top `MANUAL_TOP_N`, by number of shows). Look each up (venue website or Wikipedia), type the number in the `capacity` column, save, and re-run this notebook. Blank rows are ignored. The file is never overwritten once it exists.
- **Proxy:** `median_max_listings` (from section 2) is saved for every venue, so there is always some size signal, even where capacity is missing.

In [7]:
MANUAL_FILE = MANUAL / "venue_capacity_manual.csv"
base = venues.merge(sgv, on="venue_key", how="left")

if MANUAL_FILE.exists():
    manual = pd.read_csv(MANUAL_FILE)
    manual = manual[pd.to_numeric(manual["capacity"], errors="coerce").notna()][["venue_key", "capacity"]]
    print(f"Manual capacities loaded: {len(manual)}")
else:
    manual = pd.DataFrame(columns=["venue_key", "capacity"])
    if LIMIT:
        print("LIMIT is set: not writing the hand-fill file (it needs the full venue list)")
    else:
        need = base[base["sg_capacity"].isna() & (base["sg_country"].fillna("US") == "US")].head(MANUAL_TOP_N)
        need[["venue_key", "venue", "city", "sg_state", "n_shows"]].assign(capacity="").to_csv(MANUAL_FILE, index=False)
        print(f"Wrote {MANUAL_FILE} with {len(need)} venues to fill in by hand "
              f"({need['n_shows'].sum() / venues['n_shows'].sum():.0%} of all shows)")

Manual capacities loaded: 0


## 7. Combine and save

In [8]:
out = (base.merge(manual.rename(columns={"capacity": "manual_capacity"}), on="venue_key", how="left")
           .merge(census, on="venue_key", how="left"))
out["capacity"] = pd.to_numeric(out["manual_capacity"], errors="coerce").fillna(out["sg_capacity"])
out["capacity_source"] = np.select([out["manual_capacity"].notna(), out["sg_capacity"].notna()],
                                   ["manual", "seatgeek"], "missing")
out["venue_tier"] = pd.cut(out["capacity"], [0, 500, 2500, np.inf],
                           labels=["club (500 or less)", "theater (501-2,500)", "large (2,500+)"]).astype(str)
out.loc[out["capacity"].isna(), "venue_tier"] = "unknown"
# US flag: True / False when SeatGeek matched the venue, blank when we could not tell
out["is_us"] = out["sg_country"].map(lambda c: pd.NA if pd.isna(c) else c == "US")
out["country"] = out["sg_country"]
out["state"] = out["sg_state"]

cols = ["venue_key", "venue", "city", "state", "country", "is_us", "lat", "lon", "timezone", "n_shows",
        "median_price_min", "capacity", "capacity_source", "venue_tier", "median_max_listings",
        "sg_venue_id", "sg_name", "match_sim", "sg_venue_score", "sg_upcoming_events",
        "county_fips", "county_name", "median_income", "county_population"]
out = out[cols]
out.to_csv(OUT_PATH, index=False)

in_shows = lambda m: shows["venue_key"].isin(out.loc[m, "venue_key"]).mean()
print(f"Saved {OUT_PATH}: {len(out):,} venues")
print(f"  matched to SeatGeek: {out['sg_venue_id'].notna().mean():.0%} of venues, {in_shows(out['sg_venue_id'].notna()):.0%} of shows")
print(f"  US venues:           {(out['is_us'] == True).sum():,} | non-US: {(out['is_us'] == False).sum():,} "
      f"| unknown: {out['is_us'].isna().sum():,} ({in_shows(out['is_us'].isna()):.0%} of shows)")
print(f"  capacity known:      {out['capacity'].notna().mean():.0%} of venues, {in_shows(out['capacity'].notna()):.0%} of shows")
print(f"  county income:       {out['median_income'].notna().mean():.0%}")
out["venue_tier"].value_counts().to_frame("venues")

Saved data\venues.csv: 2,722 venues
  matched to SeatGeek: 78% of venues, 75% of shows
  US venues:           1,943 | non-US: 188 | unknown: 591 (25% of shows)
  capacity known:      17% of venues, 19% of shows
  county income:       71%


,venues
venue_tier,
unknown,2261
"large (2,500+)",345
"theater (501-2,500)",109
club (500 or less),7


In [9]:
# Busiest venues SeatGeek could not match: check by hand (wrong city spelling? a new or closed venue?)
out[out["sg_venue_id"].isna()][["venue", "city", "n_shows", "median_max_listings"]].head(20)

,venue,city,n_shows,median_max_listings
2,House of Blues,Houston,93,1.0
6,Birdland,New York,81,1.0
14,The Hi Fi,Indianapolis,72,1.0
18,Eagles Ballroom,Milwaukee,67,1.0
24,History,Toronto,57,2.0
26,Revolution Hall,Portland,56,9.0
29,The Intersection,Grand Rapids,55,1.0
31,The Sinclair,Cambridge,55,1.0
40,Greek Theatre,Los Angeles,51,30.0
44,First Avenue,Minneapolis,51,1.0


In [10]:
# Large venues (2,500+) are outside our small/mid-venue scope: check them before the model step drops them
out[out["venue_tier"] == "large (2,500+)"][["venue", "city", "state", "capacity", "n_shows"]].head(30)

,venue,city,state,capacity,n_shows
17,Hollywood Bowl,Los Angeles,CA,17500.0,68
22,Madison Square Garden,New York,NY,19812.0,58
95,Davies Symphony Hall,San Francisco,CA,2700.0,43
113,Hollywood Palladium,Los Angeles,CA,4000.0,41
152,Climate Pledge Arena,Seattle,WA,15354.0,36
158,Coca-Cola Roxy Theatre,Atlanta,GA,3000.0,36
159,Moody Center ATX,Austin,TX,15000.0,36
167,Fillmore Auditorium,Denver,CO,11000.0,35
183,Scotiabank Arena,Toronto,ON,19800.0,34
198,Truliant Amphitheater,Charlotte,NC,19500.0,33
